In [40]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from pathlib import Path


In [41]:
df_patients = pd.read_csv("data/data_octobre_2023/Pseudonymisation_provisoire.csv", sep=";",encoding_errors='ignore')

In [42]:
def geocode(df):
    """Geocode dataframe with Etalab addok"""
    print("Proceed to geocode on address...")
    for i in df.index:
        try:
            # get json response
            r = requests.get('https://addok-data.curie.net/search?q='+df["adresse"][i])
            response = r.json()

            if i%100==0 : 
                print(f"Proceed geocoding at the {i}th row")
            if response["features"]!=[]:
                # parse json to insert value in dataframe
                #df.at[i, 'nip'] = str(df["pseudo_provisoire"][i])
                df.at[i, 'x'] = str(response["features"][0]["geometry"]["coordinates"][0])
                df.at[i, 'y'] = str(response["features"][0]["geometry"]["coordinates"][1])
                df.at[i, 'score'] = str(response["features"][0]["properties"]["score"])

                if float(df["score"][i])<0.4:
                    df.at[i, 'trust_score'] = 'low'
                elif float(df["score"][i])>0.4 and float(df["score"][i])<0.65:
                    df.at[i, 'trust_score'] = 'middle'
                elif float(df["score"][i])>0.65 and float(df["score"][i])<0.9:
                    df.at[i, 'trust_score'] = 'middle'
                else:
                    df.at[i, 'trust_score'] = 'high'

                df.at[i, 'street'] = str(response["features"][0]["properties"]["name"]).replace("'", " ")
                df.at[i, 'city'] = str(response["features"][0]["properties"]["city"]).replace("'", " ")
                df.at[i, 'pc_city'] = str(response["features"][0]["properties"]["postcode"])
                df.at[i, 'ic_city'] = str(response["features"][0]["properties"]["citycode"])

                context = (str(response["features"][0]["properties"]["context"]).replace("'", " ")).split(",")
                df.at[i, 'code_dept'] = context[0]
                df.at[i, 'dept'] = context[1]

                if len(df["code_dept"][i])==2:
                    df.at[i, 'reg'] = context[2]
                else:
                    df.at[i, 'reg'] = "other"
                #df.at[i, 'code_country'] = str(df["pays"][i])
                df.at[i, 'address'] = str(response["features"][0]["properties"]["label"]).replace("'", " ")
                
                if (df["requete"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_init'] = "true"
                else:
                    df.at[i, 'address_has_num_init'] = "false"
                    
                if (df["street"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_geoloc'] = "true"
                else:
                    df.at[i, 'address_has_num_geoloc'] = "false"
                    
                if df["codepost"][i] == df["pc_city"][i]:
                    df.at[i, 'same_city'] = "true"
                else:
                    df.at[i, 'same_city'] = "false" 
                    
                if "hotel" in df["street"][i] or "hôtel" in df["street"][i]: 
                    df.at[i, 'hostel'] = "true"
                else:
                    df.at[i, 'hostel'] = "false"

                if "chez" in df["street"][i]: 
                    df.at[i, 'hosted'] = "true"
                else:
                    df.at[i, 'hosted'] = "false"
                    
                df.at[i, 'date_geoloc'] = str(datetime.date.today())
                df.at[i, 'etalab_version'] = str(response["licence"])
                df.at[i, 'ban_version'] = "2021-04-27"
            else:
                pass
        except:
            pass
    return df
    
def spatialjoin(df, df_iris, df_epci, df_dept):
    """Spatial join of the database and the iris/epci layers"""
    print("Perfom to spatial join on IRIS and EPCI layers...")
    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=2154)

    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)

    df_epci =df_epci.to_crs(epsg=2154)
    df_join_iris_epci = gpd.sjoin(df_join_iris, df_epci[['CODE_EPCI','geometry']], how="left", op='within')
    df_join_iris_epci.drop('index_right', axis=1, inplace=True)

    df_join_iris_epci_dept = gpd.sjoin(df_join_iris_epci, df_dept[['CODE_DEPT','geometry']], how="left", op='within')
    df_join_iris_epci_dept.drop('index_right', axis=1, inplace=True)
    return df_join_iris_epci_dept



In [43]:
#df_geocoded = geocode(df_patients)
#df_geocoded.to_csv("H:/canc_air/data/data_octobre_2023/Pseudonymisation_provisoire_geocoded.csv", sep=";")
df_geocoded = pd.read_csv("data/data_octobre_2023/Pseudonymisation_provisoire_geocoded_spatial.csv", sep=";")
df_geocoded.columns

Index(['Unnamed: 0', 'pseudo_provisoire', 'adresse', 'codepost',
       'nom_commune_postal', 'requete', 'x', 'y', 'score', 'trust_score',
       'street', 'city', 'pc_city', 'ic_city', 'code_dept', 'dept', 'reg',
       'address', 'address_has_num_init', 'address_has_num_geoloc',
       'same_city', 'hostel', 'hosted', 'date_geoloc', 'geometry', 'CODE_IRIS',
       'CODE_EPCI', 'CODE_DEPT'],
      dtype='object')

In [44]:
df_iris = gpd.read_file('data/zones_geographiques/iris/CONTOURS-IRIS.shp')
df_epci = gpd.read_file('data/zones_geographiques/epci/EPCI_SHAPEFILE.shp')
df_dept = gpd.read_file('data/zones_geographiques/departements/DEPARTEMENT.shp')

In [45]:
df_geocoded.drop(['Unnamed: 0'], axis=1, inplace=True)
df_geocoded.head()

,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,trust_score,street,...,address_has_num_init,address_has_num_geoloc,same_city,hostel,hosted,date_geoloc,geometry,CODE_IRIS,CODE_EPCI,CODE_DEPT
0,1,34 RUE DES FRERES CHAUSSONS,92600,ASNIERES-SUR-SEINE,34 RUE DES FRERES CHAUSSONS 92600 ASNI...,2.289499,48.916298,0.832367,middle,34 Rue des Frères Chausson,...,True,True,True,False,False,2024-01-05,POINT (647928.3162985359 6868712.859312387),920040302,200054781.0,92
1,2,11 RUE EMILE DUBOIS,75014,PARIS,11 RUE EMILE DUBOIS 75014 PARIS,2.336628,48.831707,0.972567,high,11 Rue Emile Dubois,...,True,True,True,False,False,2024-01-05,POINT (651303.2326338619 6859276.896689738),751145406,200054781.0,75
2,3,48 CHEMIN VERT,78680,EPONE,48 CHEMIN VERT 78680 EPONE,1.797376,48.950412,0.960861,high,48 Chemin Vert,...,True,True,True,False,False,2024-01-05,POINT (611921.2622170823 6872942.907671936),782170102,200059889.0,78
3,4,18 ALLEE DE LA CHARNILLE,47140,SAINT-SYLVESTRE-SUR-LOT,18 ALLEE DE LA CHARNILLE 47140 SAIN...,0.809474,44.404892,0.805540,middle,18 Allée de la Charmille,...,True,True,True,False,False,2024-01-05,POINT (525576.5290254143 6369736.268092031),472800000,200068930.0,47
4,5,31 RUE DU GENERAL DE MIRIBEL,92500,RUEIL-MALMAISON,31 RUE DU GENERAL DE MIRIBEL 92500 RUEI...,2.173326,48.865232,0.973612,high,31 Rue du Général de Miribel,...,True,True,True,False,False,2024-01-05,POINT (639354.9912859926 6863117.583096649),920630504,200054781.0,92


In [46]:
df_geocoded_spatial= spatialjoin(df_geocoded, df_iris, df_epci, df_dept)
df_geocoded_spatial.to_csv("../../analyse_clinique/Resultats/Excels/Pseudonymisation_provisoire_geocoded_spatial.csv", sep=";")

Perfom to spatial join on IRIS and EPCI layers...


c:\Users\lpokambo\.conda\envs\Loice\lib\site-packages\IPython\core\interactiveshell.py:3577: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\lpokambo\.conda\envs\Loice\lib\site-packages\IPython\core\interactiveshell.py:3577: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\lpokambo\.conda\envs\Loice\lib\site-packages\IPython\core\interactiveshell.py:3577: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
df_geocoded_spatial_wout_na_adresse = df_geocoded_spatial.dropna(subset="adresse")

In [48]:
df_dept =  gpd.read_file('data/zones_geographiques/departements/DEPARTEMENT.shp')
df_france = df_dept.dissolve()

In [49]:
patients_in_france = gpd.sjoin(df_geocoded_spatial_wout_na_adresse, df_france, how='inner', op='intersects')


c:\Users\lpokambo\.conda\envs\Loice\lib\site-packages\IPython\core\interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [50]:
print(f"Le nombre de patients totaux : {len(df_geocoded_spatial)}")
print(f"Le nombre de patients avec une adresse : {len(df_geocoded_spatial)-df_geocoded_spatial['adresse'].isnull().sum()}. verification : {len(df_geocoded_spatial_wout_na_adresse)}")
print(f"Le nombre de patients présent en france métropolitaine : {len(patients_in_france)}")
print(f"Le nombre de patients comprennant un EPCI, Iris et ")

Le nombre de patients totaux : 64294
Le nombre de patients avec une adresse : 63146. verification : 63146
Le nombre de patients présent en france métropolitaine : 61911
Le nombre de patients comprennant un EPCI, Iris et 
